##  Text to Vectors and Semantic Similarity

An **embedding** is a dense vector (ordered list of numbers) produced by a model such that texts with **similar meaning** map to **nearby** points in vector space.

**How Text Becomes a Vector:**
1. Raw text
2. Tokenization
3. Encoding (embedding model)
4. Store or compare

Let's preview how a sentence is converted into an embedding using `SentenceTransformer`.

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # Same model used in the Chroma lab below


sentence_a = "Refunds are processed within 5 to 7 business days"  # FAQ-style stored text
sentence_b = "When will I get my money back after returning the item?"  # User-style paraphrase

vectora=model.encode(sentence_a,convert_to_numpy=True)
vectorb=model.encode(sentence_b,convert_to_numpy=True)

print("The length of vectora : ",len(vectora))
print("First five vectors : ",vectora[:5])



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5848.77it/s]


The length of vectora :  384
First five vectors :  [-0.06581643 -0.00359726  0.08447061  0.01167005  0.04429862]


## 3. Prepare Sample Data

Good retrieval starts with clean, small chunks — one idea per row.
Our example is an **e-commerce support knowledge base** with five short FAQs.

In [8]:
records = [  # Each dict = one Chroma row (id, text, metadata)
    {"id": "doc1", "text": "Customers can return products within 30 days of delivery.", "metadata": {"category": "returns"}},
    {"id": "doc2", "text": "Refunds are processed within 5 to 7 business days after the return is approved.", "metadata": {"category": "returns"}},
    {"id": "doc3", "text": "Orders above 499 rupees qualify for free shipping.", "metadata": {"category": "shipping"}},
    {"id": "doc4", "text": "You can reset your password from the account settings page.", "metadata": {"category": "account"}},
    {"id": "doc5", "text": "Express delivery orders usually arrive within 24 to 48 hours.", "metadata": {"category": "shipping"}},
]

# Note: Each dict is one record — like one SQL row.

## 4. Create the Chroma Client and Collection
We will create a Persistent Client (saves to disk) and a new Collection (like an SQL Table) to store our embeddings.

In [9]:
# creating the client and the collection 
import chromadb
from pprint import pprint

client=chromadb.PersistentClient(path="./chroma_store")

collection=client.get_or_create_collection(
    name="Support_knowledge_base",
    embedding_function=None # We will pass embeddings manually

)

print("Collection : ",collection.name)
print('cont before usert : ',collection.count())

Collection :  Support_knowledge_base
cont before usert :  0
